# GramCast — Residual U-Net training sweep (Kaggle GPU)

SIH26074. Trains the P10/P50/P90 residual U-Net on the already-built, already-verified training pairs (see `DEV_LOG.md` in the repo for the full verification history — every piece here was tested locally on CPU before this notebook was written).

**This notebook runs an automated sweep of several training trials in one go** (defined in the `TRIALS` list in the config cell below) — each trial trains a fresh model from scratch with one hyperparameter changed from the baseline, so you don't need to manually edit and re-run cells between trials. Just run the whole notebook once (or click **Save Version → Save & Run All**) and every trial runs back to back, ending with one zip containing every trial's checkpoints and logs.

**⚠️ Re-zip and re-upload your dataset before running this version.** As of 2026-09-26, the input tensor includes 3 new ERA5 humidity/wind channels (`era5_humidity_wind.npz`, 17 channels total instead of 14) — if your existing Kaggle Dataset predates this, it won't have that file and the data-loading cell will raise a clear `FileNotFoundError` telling you so.

**Before running this notebook:**
1. On your local machine, zip `data/processed/training_pairs/` (excluding `zonal_stats_*.csv`, that's a demo output, not training input) — it's `fine_rainfall.npy`, `coarse_rainfall_bicubic.npy`, `terrain_static.npz`, `split_indices.npz`, `valid_mask.npy`, `dates.npy`, and now also `era5_humidity_wind.npz`.
2. Go to kaggle.com → Datasets → your existing dataset (or New Dataset) → upload the new zip as a new version (or a fresh dataset) → keep it **Private** → Create/Update.
3. In this notebook, click **Add Input** (right panel) → search for your dataset → add it (or re-add it if it updated to a new version).
4. **Settings → Accelerator → GPU**, pick a **single-GPU** option (P100 or T4 x1) — this notebook's code is single-GPU only, see the note the device-check cell prints if you pick a x2 option anyway. Without any GPU the notebook still runs, just on CPU, defeating the point.
5. **Settings → Internet → Off** is safe here — nothing in this notebook makes an external call, all data comes from the Dataset input you added in step 3. Good practice to confirm that rather than assume it.
6. Check your remaining weekly GPU quota (free tier caps it, shown in Settings) before kicking off a run — each trial is small and fast (the first real run here took ~2 minutes for 52 epochs), so a 4-trial sweep is on the order of 10-15 minutes of GPU time, well inside a normal quota.
7. `DATA_DIR` (config cell below) is auto-discovered by searching `/kaggle/input` for `fine_rainfall.npy`, so it doesn't matter what you named the dataset or whether Kaggle nested it under the zipped folder's own name (`/kaggle/input/<your-dataset-name>/training_pairs/...` is just as fine as a flat `/kaggle/input/<your-dataset-name>/...`) — no manual path editing needed. It raises a clear `FileNotFoundError` listing what it actually found under `/kaggle/input` if the dataset input is missing or misnamed.
8. **To change what the sweep tests:** edit the `TRIALS` list in the config cell — each entry is one full from-scratch run. Add, remove, or edit entries; trial names must be unique (checked automatically).

**On persistence — read this before a long run:** `/kaggle/working/` (where checkpoints and logs go) only survives past an interactive session if you actually **commit** it.
- **Interactive** (editor open, watching progress live): if your browser tab/session just closes without committing, whatever's in `/kaggle/working/` can be lost. Click **Save Version → Quick Save** periodically to snapshot the current checkpoints/logs as notebook output you can always come back to.
- **Save Version → Save & Run All (Commit)** (recommended): runs the *entire* sweep end-to-end in the background on Kaggle's servers, independent of your browser being open. You can close the tab and come back later — check progress under the **Version history** tab, and the final `/kaggle/working/` output (every trial's checkpoints + logs) is preserved as that version's output.
- **If a run gets interrupted mid-sweep:** unlike a single long run, there's no `RESUME_FROM` here — each trial is short enough (~2 min) that simply re-running the whole sweep from scratch is cheaper than building resume logic for it. Any trials that *did* finish before the interruption already have their checkpoints saved in `/kaggle/working/checkpoints/` and are safe; just re-run to fill in the rest (existing files won't be overwritten, since every trial's `run_name` is unique per run).

**After the sweep finishes:** the last cell before the zip prints a ranked comparison table (`best_val_loss` per trial). That ranks the *training* objective only — download every trial's `*_best.pt` and run `ml/evaluate.py` on each locally to see which one actually wins on the real metrics that matter (RMSE, POD/FAR/CSI, etc.), since a lower val_loss doesn't automatically mean better real-world performance.

In [ ]:
import os, sys, time, csv
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected. Settings -> Accelerator -> GPU, then restart.")
elif torch.cuda.device_count() > 1:
    print(
        f"NOTE: {torch.cuda.device_count()} GPUs visible (e.g. 'GPU T4 x2'), but this notebook is single-GPU "
        "code - it will only use GPU 0, the second GPU sits idle. Not a bug, just not using all the hardware "
        "you selected. If you want to use both, pick a single-GPU accelerator ('P100' or 'T4 x1') instead, or "
        "wrap the model in nn.DataParallel (adds real complexity for a model this small - probably not worth it)."
    )


def fmt_time(seconds: float) -> str:
    """e.g. 3725.4 -> '1h 2m 5s' - used for live elapsed/ETA display."""
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"

In [ ]:
# --- Config ---
def _find_training_pairs_dir(root: Path = Path("/kaggle/input")) -> Path:
    """Locate the directory containing fine_rainfall.npy under /kaggle/input. Deliberately NOT a
    hardcoded path: zipping a folder and uploading it to Kaggle preserves that folder's name inside
    the mount (training_pairs/fine_rainfall.npy -> /kaggle/input/<dataset-slug>/training_pairs/...,
    one level deeper than a flat zip would), and the dataset slug itself depends on what you named
    it - both are real, common causes of FileNotFoundError here, not a data problem."""
    matches = list(root.rglob("fine_rainfall.npy"))
    if not matches:
        listing = chr(10).join(f"  {p}" for p in root.rglob("*")) or "  (nothing under /kaggle/input - did you Add Input for the dataset?)"
        raise FileNotFoundError(f"Could not find fine_rainfall.npy anywhere under {root}. Contents:" + chr(10) + listing)
    if len(matches) > 1:
        raise FileNotFoundError(f"Found fine_rainfall.npy in more than one place, ambiguous: {matches}")
    return matches[0].parent


DATA_DIR = _find_training_pairs_dir()
print(f"Resolved DATA_DIR = {DATA_DIR}")
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
LOG_DIR = Path("/kaggle/working/logs")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 300  # a safety CAP per trial, not a target - PATIENCE below is what actually decides when a trial stops
PATIENCE = 15  # stop a trial if its val_loss hasn't improved for this many consecutive epochs
BATCH_SIZE = 32
SEED = 42  # re-applied at the START of every trial below, so each trial's weight init is controlled the same way
DETERMINISTIC = True  # see note below

# Automated trial sweep: each dict is one full from-scratch training run, varying exactly one thing
# from the baseline so results stay comparable. Edit this list to change what the sweep tests - no
# manual cell-editing between trials needed, Save & Run All does the whole sweep unattended.
TRIALS = [
    {"name": "baseline", "base_channels": 16, "lr": 1e-3, "heavy_weight": 4.0},
    {"name": "heavy_weight_8", "base_channels": 16, "lr": 1e-3, "heavy_weight": 8.0},
    {"name": "base_channels_32", "base_channels": 32, "lr": 1e-3, "heavy_weight": 4.0},
    {"name": "lr_5e-4", "base_channels": 16, "lr": 5e-4, "heavy_weight": 4.0},
]
_trial_names = [t["name"] for t in TRIALS]
assert len(set(_trial_names)) == len(_trial_names), f"duplicate trial names in TRIALS, must be unique: {_trial_names}"

# GPU non-determinism gotcha: we verified strict reproducibility (same seed -> byte-identical results) on CPU
# earlier this project (see DEV_LOG 2026-09-25) - but PyTorch's GPU convolutions are NOT deterministic by
# default even with a fixed seed, because cuDNN picks convolution algorithms opportunistically for speed. That
# CPU-verified guarantee would silently NOT carry over to this GPU run unless forced explicitly:
if DETERMINISTIC:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # costs some speed - deterministic algorithms aren't always the fastest ones
else:
    torch.backends.cudnn.benchmark = True  # faster, but two runs with the same seed can differ slightly

## Load the pre-built, pre-verified training pairs

In [ ]:
fine = np.load(DATA_DIR / "fine_rainfall.npy")
coarse_bicubic = np.load(DATA_DIR / "coarse_rainfall_bicubic.npy")
splits = np.load(DATA_DIR / "split_indices.npz")
terrain = dict(np.load(DATA_DIR / "terrain_static.npz"))
valid_mask = np.load(DATA_DIR / "valid_mask.npy")

era5_path = DATA_DIR / "era5_humidity_wind.npz"
if not era5_path.exists():
    raise FileNotFoundError(
        f"{era5_path} not found. This dataset was uploaded before ERA5 humidity/wind was added "
        "(2026-09-26) - re-zip data/processed/training_pairs/ locally (it now includes "
        "era5_humidity_wind.npz alongside the files you already had) and re-upload/update the "
        "Kaggle Dataset, then Add Input again if it created a new version."
    )
era5_npz = np.load(era5_path)
era5 = {name: era5_npz[name] for name in ["humidity_proxy", "wind_u", "wind_v"]}

print(f"fine: {fine.shape}, coarse_bicubic: {coarse_bicubic.shape}")
print(f"train/val/test: {len(splits['train'])}/{len(splits['val'])}/{len(splits['test'])} (expect 4636/244/702)")
assert fine.shape == (5582, 35, 50), f"Unexpected shape {fine.shape} - is this really the verified GramCast dataset?"
assert not np.isnan(fine).any() and not np.isnan(coarse_bicubic).any(), "NaN in loaded data - re-check the upload"
for name, arr in era5.items():
    assert arr.shape == fine.shape, f"era5['{name}'] shape {arr.shape} doesn't match fine {fine.shape}"
    assert not np.isnan(arr).any(), f"NaN in era5['{name}'] - re-check the upload"
print(f"era5 channels loaded: {list(era5.keys())}, all shape {fine.shape}, no NaN")

## Model code (inlined from `ml/model/`, verified locally — see `DEV_LOG.md`)

Kept identical to the tested local modules rather than reimplemented, to avoid any transcription drift from what was actually verified.

In [ ]:
# --- preprocessing.py ---
WORLDCOVER_CLASSES = [10, 20, 30, 40, 50, 60, 80, 90, 95]
ERA5_CHANNELS = ["humidity_proxy", "wind_u", "wind_v"]


def compute_normalization_stats(terrain, coarse_bicubic_train, era5_train):
    stats = {
        "dem_mean": float(terrain["dem"].mean()), "dem_std": float(terrain["dem"].std()),
        "slope_mean": float(terrain["slope"].mean()), "slope_std": float(terrain["slope"].std()),
        "rain_mean": float(coarse_bicubic_train.mean()), "rain_std": float(coarse_bicubic_train.std()),
    }
    for name in ERA5_CHANNELS:
        arr = era5_train[name]
        stats[f"{name}_mean"] = float(arr.mean())
        stats[f"{name}_std"] = float(arr.std())
    return stats


def build_input_tensor(coarse_bicubic, terrain, stats, era5):
    """Channel order: [rain, humidity_proxy, wind_u, wind_v, dem, slope, aspect_sin, aspect_cos, *worldcover_onehot]"""
    T, H, W = coarse_bicubic.shape
    for name in ERA5_CHANNELS:
        if era5[name].shape != (T, H, W):
            raise ValueError(f"era5['{name}'] shape {era5[name].shape} doesn't match coarse_bicubic's {(T, H, W)}")

    rain_norm = (coarse_bicubic - stats["rain_mean"]) / stats["rain_std"]
    dem_norm = (terrain["dem"] - stats["dem_mean"]) / stats["dem_std"]
    slope_norm = (terrain["slope"] - stats["slope_mean"]) / stats["slope_std"]
    aspect_rad = np.deg2rad(terrain["aspect"])
    aspect_sin = np.sin(aspect_rad)
    aspect_cos = np.cos(aspect_rad)

    unknown_mask = ~np.isin(terrain["worldcover"], WORLDCOVER_CLASSES)
    if unknown_mask.any():
        unknown_classes = sorted(set(terrain["worldcover"][unknown_mask].tolist()))
        raise ValueError(f"WorldCover class(es) {unknown_classes} not in WORLDCOVER_CLASSES {WORLDCOVER_CLASSES}")

    time_varying_channels = [rain_norm] + [(era5[name] - stats[f"{name}_mean"]) / stats[f"{name}_std"] for name in ERA5_CHANNELS]
    static_channels = [dem_norm, slope_norm, aspect_sin, aspect_cos]
    for cls in WORLDCOVER_CLASSES:
        static_channels.append((terrain["worldcover"] == cls).astype("float32"))

    n_channels = len(time_varying_channels) + len(static_channels)
    tensor = np.empty((T, n_channels, H, W), dtype="float32")
    for i, ch in enumerate(time_varying_channels):
        tensor[:, i, :, :] = ch
    offset = len(time_varying_channels)
    for i, ch in enumerate(static_channels):
        tensor[:, offset + i, :, :] = np.tile(ch, (T, 1, 1))
    return tensor

In [ ]:
# --- nearest_neighbor.py (predict only, for the mass-conservation target) ---
def nn_predict(fine_shape_grid):
    T, h, w = fine_shape_grid.shape
    ch, cw = h // 5, w // 5
    coarse = fine_shape_grid.reshape(T, ch, 5, cw, 5).mean(axis=(2, 4))
    return np.repeat(np.repeat(coarse, 5, axis=1), 5, axis=2)

In [ ]:
# --- mass_conservation.py ---
COARSEN_FACTOR = 5


class MassConservation(nn.Module):
    def forward(self, fine_pred, coarse_value):
        B, C, H, W = fine_pred.shape
        ch, cw = H // COARSEN_FACTOR, W // COARSEN_FACTOR

        fine_blocks = fine_pred.reshape(B, C, ch, COARSEN_FACTOR, cw, COARSEN_FACTOR)
        pred_block_mean = fine_blocks.mean(dim=(3, 5), keepdim=True)

        coarse_blocks = coarse_value.reshape(B, C, ch, COARSEN_FACTOR, cw, COARSEN_FACTOR)
        target_block_mean = coarse_blocks.mean(dim=(3, 5), keepdim=True)

        eps = 1e-6
        safe_pred_mean = torch.where(pred_block_mean > eps, pred_block_mean, torch.full_like(pred_block_mean, eps))
        scale = target_block_mean / safe_pred_mean

        rescaled_blocks = fine_blocks * scale
        near_zero_pred = pred_block_mean <= eps
        uniform_fallback = target_block_mean.expand_as(fine_blocks)
        rescaled_blocks = torch.where(near_zero_pred.expand_as(fine_blocks), uniform_fallback, rescaled_blocks)

        return rescaled_blocks.reshape(B, C, H, W)

In [ ]:
# --- unet.py --- P10/P50/P90 quantile heads, monotonicity guaranteed by construction (see DEV_LOG 2026-09-25)
PAD_H, PAD_W = 36, 52


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualUNet(nn.Module):
    def __init__(self, in_channels, base_channels=16):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_channels * 2, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = ConvBlock(base_channels * 2, base_channels)
        self.correction_head = nn.Conv2d(base_channels, 3, 1)
        self.mass_conservation = MassConservation()

    def forward(self, x):
        B, C, H, W = x.shape
        pad_h, pad_w = PAD_H - H, PAD_W - W
        x_padded = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        e1 = self.enc1(x_padded)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        out_padded = self.correction_head(d1)
        return out_padded[:, :, :H, :W]

    def predict_quantiles(self, x, bicubic_raw, coarse_nn_raw):
        raw = self.forward(x)
        p50_correction = raw[:, 0:1, :, :]
        delta_low = F.softplus(raw[:, 1:2, :, :])
        delta_high = F.softplus(raw[:, 2:3, :, :])
        p50_raw = torch.clamp(bicubic_raw + p50_correction, min=0.0)
        p50 = self.mass_conservation(p50_raw, coarse_nn_raw)
        p10 = torch.clamp(p50 - delta_low, min=0.0)
        p90 = p50 + delta_high
        return {"p10": p10, "p50": p50, "p90": p90}

In [ ]:
# --- quantile_loss.py --- verified against pinball loss's asymmetric-penalty property, see DEV_LOG 2026-09-25
HEAVY_THRESHOLD_MM = 64.5
HEAVY_WEIGHT = 4.0
QUANTILES = {"p10": 0.1, "p50": 0.5, "p90": 0.9}


def pinball_loss(pred, truth, tau):
    diff = truth - pred
    return torch.maximum(tau * diff, (tau - 1) * diff)


class QuantileHeavyRainLoss(nn.Module):
    def __init__(self, heavy_threshold=HEAVY_THRESHOLD_MM, heavy_weight=HEAVY_WEIGHT):
        super().__init__()
        self.heavy_threshold = heavy_threshold
        self.heavy_weight = heavy_weight

    def forward(self, preds, truth, mask=None):
        log_truth = torch.log1p(truth)
        weight = 1.0 + self.heavy_weight * (truth >= self.heavy_threshold).float()
        total = 0.0
        for name, tau in QUANTILES.items():
            log_pred = torch.log1p(preds[name])
            pin = pinball_loss(log_pred, log_truth, tau) * weight
            if mask is not None:
                pin = pin[:, :, mask]
            total = total + pin.mean()
        return total / len(QUANTILES)

## Sanity check the inlined model code before training (cheap, catches a transcription error instantly)

In [ ]:
era5_train = {name: arr[splits["train"]] for name, arr in era5.items()}
stats = compute_normalization_stats(terrain, coarse_bicubic[splits["train"]], era5_train)
era5_head = {name: arr[:4] for name, arr in era5.items()}
_x = torch.from_numpy(build_input_tensor(coarse_bicubic[:4], terrain, stats, era5_head))
_bicubic = torch.from_numpy(coarse_bicubic[:4]).unsqueeze(1).float()
_coarse_nn = torch.from_numpy(nn_predict(fine[:4])).unsqueeze(1).float()

_model = ResidualUNet(in_channels=_x.shape[1], base_channels=16)  # fixed value here - this is just a code-correctness check, independent of which base_channels the sweep trials use
_model.eval()
with torch.no_grad():
    _q = _model.predict_quantiles(_x, _bicubic, _coarse_nn)

assert (_q["p10"] <= _q["p50"]).all() and (_q["p50"] <= _q["p90"]).all(), "Monotonicity broken - do not train, check the inlined code above against ml/model/unet.py"
assert not any(torch.isnan(_q[k]).any() for k in _q), "NaN in a fresh forward pass - do not train"
print(f"Inlined model code sanity check passed: shapes ({_x.shape[1]} input channels), monotonicity, and NaN-freedom all confirmed on a real batch.")
del _x, _bicubic, _coarse_nn, _model, _q, era5_head

## Dataset + DataLoaders

In [ ]:
class GramCastDataset(Dataset):
    def __init__(self, indices, fine, coarse_bicubic, terrain, stats, era5):
        self.indices = indices
        self.fine = fine[indices]
        self.coarse_bicubic = coarse_bicubic[indices]
        self.coarse_nn = nn_predict(self.fine)
        era5_sliced = {name: arr[indices] for name, arr in era5.items()}
        self.input_tensor = build_input_tensor(self.coarse_bicubic, terrain, stats, era5_sliced)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return {
            "x": torch.from_numpy(self.input_tensor[i]),
            "bicubic_raw": torch.from_numpy(self.coarse_bicubic[i]).unsqueeze(0).float(),
            "coarse_nn_raw": torch.from_numpy(self.coarse_nn[i]).unsqueeze(0).float(),
            "truth": torch.from_numpy(self.fine[i]).unsqueeze(0).float(),
        }


train_ds = GramCastDataset(splits["train"], fine, coarse_bicubic, terrain, stats, era5)
val_ds = GramCastDataset(splits["val"], fine, coarse_bicubic, terrain, stats, era5)
test_ds = GramCastDataset(splits["test"], fine, coarse_bicubic, terrain, stats, era5)

# num_workers=0: all data is already in memory (loaded via np.load above), so extra worker processes give
# minimal speedup here and add a platform-dependent multiprocessing risk (confirmed locally on Windows - a
# freeze_support error - not worth risking on an unverified Kaggle/Linux run for negligible benefit).
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}, input channels={train_ds.input_tensor.shape[1]}")

## Training loop — runs every trial in `TRIALS` back to back, checkpointing every epoch

In [ ]:
def run_epoch(model, loader, loss_fn, mask, device, optimizer=None, desc=""):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        x = batch["x"].to(device)
        bicubic_raw = batch["bicubic_raw"].to(device)
        coarse_nn_raw = batch["coarse_nn_raw"].to(device)
        truth = batch["truth"].to(device)
        with torch.set_grad_enabled(is_train):
            preds = model.predict_quantiles(x, bicubic_raw, coarse_nn_raw)
            loss = loss_fn(preds, truth, mask)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{total_loss/n_batches:.4f}")
    return total_loss / n_batches

In [ ]:
def run_trial(trial_cfg, mask, train_loader, val_loader, device):
    """Train one full model from scratch with this trial's hyperparameters. Re-seeds at the start of
    every trial so each one's weight initialization is controlled the same way - isolates the
    hyperparameter actually being varied as the only difference between trials, not also random-init luck."""
    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    run_name = time.strftime("gramcast_%Y%m%d_%H%M%S_") + trial_cfg["name"]
    base_channels = trial_cfg["base_channels"]
    lr = trial_cfg["lr"]
    heavy_weight = trial_cfg["heavy_weight"]

    model = ResidualUNet(in_channels=train_ds.input_tensor.shape[1], base_channels=base_channels).to(device)
    loss_fn = QuantileHeavyRainLoss(heavy_weight=heavy_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    epochs_since_improvement = 0
    log_path = LOG_DIR / f"{run_name}.csv"
    epoch_durations = []
    training_start = time.time()
    stopped_early = False
    final_epoch = 0

    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "lr", "batch_size", "base_channels", "heavy_weight", "seed", "timestamp", "elapsed_s"])

        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()
            train_loss = run_epoch(model, train_loader, loss_fn, mask, device, optimizer, desc=f"[{trial_cfg['name']}] epoch {epoch}/{EPOCHS} [train]")
            val_loss = run_epoch(model, val_loader, loss_fn, mask, device, optimizer=None, desc=f"[{trial_cfg['name']}] epoch {epoch}/{EPOCHS} [val]")
            elapsed = time.time() - t0
            epoch_durations.append(elapsed)

            total_elapsed = time.time() - training_start
            avg_epoch_time = sum(epoch_durations[-5:]) / len(epoch_durations[-5:])
            eta_seconds = avg_epoch_time * (EPOCHS - epoch)

            improved = val_loss < best_val_loss
            if improved:
                epochs_since_improvement = 0
            else:
                epochs_since_improvement += 1

            print(
                f"[{run_name}] epoch {epoch}/{EPOCHS}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}"
                f"{' (new best)' if improved else f' (no improvement in {epochs_since_improvement}/{PATIENCE})'} "
                f"({elapsed:.1f}s this epoch) | elapsed so far: {fmt_time(total_elapsed)} | "
                f"ETA: {fmt_time(eta_seconds)} (avg {avg_epoch_time:.1f}s/epoch)"
            )
            writer.writerow([epoch, train_loss, val_loss, lr, BATCH_SIZE, base_channels, heavy_weight, SEED, time.strftime("%Y-%m-%d %H:%M:%S"), f"{elapsed:.1f}"])
            f.flush()

            if improved:
                best_val_loss = val_loss
                torch.save(
                    {"epoch": epoch, "model_state": model.state_dict(), "val_loss": val_loss, "stats": stats,
                     "config": {"lr": lr, "batch_size": BATCH_SIZE, "base_channels": base_channels, "heavy_weight": heavy_weight, "seed": SEED}},
                    CHECKPOINT_DIR / f"{run_name}_best.pt",
                )

            torch.save(
                {"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                 "val_loss": val_loss, "best_val_loss": best_val_loss, "epochs_since_improvement": epochs_since_improvement,
                 "stats": stats, "config": {"lr": lr, "batch_size": BATCH_SIZE, "base_channels": base_channels, "heavy_weight": heavy_weight, "seed": SEED}},
                CHECKPOINT_DIR / f"{run_name}_last.pt",
            )

            final_epoch = epoch
            if epochs_since_improvement >= PATIENCE:
                print(f"[{run_name}] Early stopping: no val_loss improvement for {PATIENCE} consecutive epochs (patience reached at epoch {epoch}).")
                stopped_early = True
                break

    total_training_time = time.time() - training_start
    stop_reason = "early stopping" if stopped_early else f"reached EPOCHS cap ({EPOCHS})"
    print(f"[{run_name}] Done ({stop_reason}). Best val_loss: {best_val_loss:.4f}. Time: {fmt_time(total_training_time)}.\n")
    return {
        "trial": trial_cfg["name"], "run_name": run_name, "best_val_loss": best_val_loss,
        "epochs_run": final_epoch, "stopped_early": stopped_early, "training_time_s": total_training_time,
        **trial_cfg,
    }


mask = torch.from_numpy(valid_mask).to(device)
trial_results = []
sweep_start = time.time()

for trial_cfg in TRIALS:
    print(f"\n{'=' * 70}\nTRIAL: {trial_cfg['name']}  {trial_cfg}\n{'=' * 70}")
    result = run_trial(trial_cfg, mask, train_loader, val_loader, device)
    trial_results.append(result)

print(f"\n{'=' * 70}\nALL {len(TRIALS)} TRIALS FINISHED in {fmt_time(time.time() - sweep_start)}\n{'=' * 70}")
print(f"{'trial':<20}{'best_val_loss':>16}{'epochs_run':>12}{'stopped_early':>16}")
for r in sorted(trial_results, key=lambda r: r["best_val_loss"]):
    print(f"{r['trial']:<20}{r['best_val_loss']:>16.4f}{r['epochs_run']:>12}{str(r['stopped_early']):>16}")

best_trial = min(trial_results, key=lambda r: r["best_val_loss"])
print(f"\nBest trial by val_loss: {best_trial['trial']} ({best_trial['run_name']}), val_loss={best_trial['best_val_loss']:.4f}")
print("NOTE: val_loss ranks the TRAINING objective, not every real-world metric - run ml/evaluate.py")
print("locally against each trial's *_best.pt to compare honestly on RMSE/POD/FAR/CSI before picking a winner.")

## Package outputs into one zip for download

Zips `checkpoints/` and `logs/` from `/kaggle/working/` into a single `gramcast_output.zip`, so you only need to download one file (Output tab) instead of digging through individual checkpoint/log files by hand.

In [ ]:
import zipfile

output_zip_path = Path("/kaggle/working/gramcast_output.zip")
with zipfile.ZipFile(output_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [CHECKPOINT_DIR, LOG_DIR]:
        for f in sorted(folder.rglob("*")):
            if f.is_file():
                zf.write(f, arcname=f.relative_to(Path("/kaggle/working")))

zipped_names = zipfile.ZipFile(output_zip_path).namelist()
n_checkpoints = sum(1 for n in zipped_names if n.endswith(".pt"))
n_logs = sum(1 for n in zipped_names if n.endswith(".csv"))
assert n_checkpoints > 0, f"No .pt checkpoints found in {CHECKPOINT_DIR} - did training actually run before this cell?"
assert n_logs > 0, f"No .csv logs found in {LOG_DIR} - did training actually run before this cell?"

print(f"Wrote {output_zip_path} ({output_zip_path.stat().st_size / 1e6:.1f} MB), {n_checkpoints} checkpoint(s) + {n_logs} log(s):")
for name in zipped_names:
    print(f"  {name}")
print("\nDownload gramcast_output.zip from the Output tab (Save Version first if this was Save & Run All) - one file, everything inside.")

## Next steps after this sweep finishes

This notebook trains every trial in `TRIALS` and checkpoints each one separately. Once real numbers exist, download `gramcast_output.zip` and, back in the local repo:
1. Extract it — `checkpoints/*.pt` into `ml/checkpoints/`, `logs/*.csv` into `ml/logs/`.
2. Run `python ml/evaluate.py --checkpoint <path>` for **each trial's `*_best.pt`** — the printed val_loss ranking above is training-objective only, not the real metrics.
3. Compare honestly against the 3 baselines and across trials — report the real result even if a trial doesn't clearly win on every metric, per this project's own standing rule.
4. Independent station validation and split-conformal calibration of the P10/P90 band are still open ML issues — see `docs/ISSUES_PLAN.md`.

None of these numbers should be written up or reported until they come from an actual run of this notebook — see DEV_LOG.md's standing rule against fabricating results.